# 05. Clustering-Map-Refine 요약

Map-Reduce / Map-Refine 은 **모든 chunk** 에 LLM 을 호출하므로 문서가 길면 시간과 비용이 많이 듭니다.

그래서 아래처럼 **대표 chunk 만 골라** 요약합니다.

1. 문서를 작은 chunk 로 나누고 **임베딩(벡터화)** 합니다.
2. **K-Means** 로 N 개의 클러스터(주제 묶음)로 나눕니다.
3. 각 클러스터의 **중심점에 가장 가까운 chunk** 를 대표로 뽑습니다.
4. 대표 chunk N 개만 **Map-Refine** 으로 요약합니다.

- 출처: [gkamradt - 5 Levels Of Summarization](https://github.com/gkamradt/langchain-tutorials/blob/main/data_generation/5%20Levels%20Of%20Summarization%20-%20Novice%20To%20Expert.ipynb)
- 추가 패키지: `scikit-learn`, `matplotlib`, `seaborn`

## 환경 설정

- API 키(`OPENAI_API_KEY`, `LANGSMITH_API_KEY` 등)는 `.env` 파일 또는 시스템 환경변수에서 읽어옵니다.
- `logging.langsmith()` 를 호출하면 실행 과정이 LangSmith 에 기록되어 체인 내부 동작을 확인할 수 있습니다.

In [ ]:
# .env 파일에 저장된 API KEY 를 환경변수로 불러옵니다.
from dotenv import load_dotenv

load_dotenv()

# LangSmith 추적 설정 (프로젝트 이름: Summary)
from langchain_teddynote import logging

logging.langsmith("Summary")

## 데이터 로드

여기서는 PDF 전체를 사용합니다. (`PyMuPDFLoader` 는 `PyPDFLoader` 보다 빠르고 한글 추출 품질이 좋습니다.)

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("../data/SPRI_AI_Brief_2023년12월호_F.pdf")
docs = loader.load()
print(f"전체 페이지 수: {len(docs)}")

In [ ]:
# 페이지 경계에 상관없이 나누기 위해 모든 페이지를 하나의 텍스트로 합칩니다.
texts = "\n\n".join(doc.page_content for doc in docs)
print(f"전체 글자 수: {len(texts)}")

## 텍스트 분할

`RecursiveCharacterTextSplitter` 로 500자 단위(100자 겹침)로 나눕니다.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
split_docs = text_splitter.split_text(texts)  # 문자열 리스트
print(f"chunk 수: {len(split_docs)}")

## 임베딩

각 chunk 를 벡터로 변환합니다. Upstage 또는 OpenAI 임베딩 중 **하나만** 실행하세요.
(두 셀을 모두 실행하면 마지막에 실행한 결과가 사용됩니다.)

In [ ]:
import numpy as np
from langchain_upstage import UpstageEmbeddings

# Upstage 임베딩 (UPSTAGE_API_KEY 필요) - 한국어 성능이 좋습니다.
embeddings = UpstageEmbeddings(model="solar-embedding-1-large-passage")

# 거리 계산을 위해 numpy 배열로 변환합니다. shape: (chunk 수, 임베딩 차원)
vectors = np.array(embeddings.embed_documents(split_docs))
vectors.shape

In [ ]:
# (대안) OpenAI 임베딩을 사용하려면 아래 주석을 해제하세요.
# from langchain_openai import OpenAIEmbeddings
#
# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
# vectors = np.array(embeddings.embed_documents(split_docs))
# vectors.shape

## K-Means 클러스터링

chunk 들을 `num_clusters` 개의 그룹으로 나눕니다. 문서 길이/주제 수에 맞게 조절하세요.

In [ ]:
from sklearn.cluster import KMeans

num_clusters = 10

kmeans = KMeans(n_clusters=num_clusters, random_state=123).fit(vectors)

# 각 chunk 가 속한 클러스터 번호
kmeans.labels_

## 클러스터 시각화 (t-SNE)

고차원 임베딩을 t-SNE 로 2차원으로 줄여, 클러스터가 어떻게 나뉘었는지 확인합니다.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE

warnings.filterwarnings("ignore")

# 한글 폰트 설정 (Windows: 맑은 고딕) 및 음수 기호 깨짐 방지
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# 임베딩을 2차원으로 축소
tsne = TSNE(n_components=2, random_state=42)
reduced_data_tsne = tsne.fit_transform(vectors)

sns.set_style("white")
plt.figure(figsize=(10, 8))
sns.scatterplot(
    x=reduced_data_tsne[:, 0],
    y=reduced_data_tsne[:, 1],
    hue=kmeans.labels_,  # 클러스터별로 색 구분
    palette="deep",
    s=100,
)
plt.xlabel("Dimension 1", fontsize=12)
plt.ylabel("Dimension 2", fontsize=12)
plt.title("Clustered Embeddings", fontsize=16)
plt.legend(title="Cluster", title_fontsize=12)
plt.gcf().patch.set_facecolor("white")
plt.tight_layout()
plt.show()

## 클러스터별 대표 chunk 선택

각 클러스터 중심점(`cluster_centers_`)과 가장 가까운(유클리드 거리 최소) chunk 를 대표로 고릅니다.

In [ ]:
closest_indices = []

for i in range(num_clusters):
    # 모든 chunk 와 i 번째 클러스터 중심 사이의 거리
    distances = np.linalg.norm(vectors - kmeans.cluster_centers_[i], axis=1)

    # 거리가 가장 짧은 chunk 의 인덱스
    closest_indices.append(int(np.argmin(distances)))

# 원문 순서대로 요약하기 위해 오름차순 정렬
selected_indices = sorted(closest_indices)
selected_indices

In [ ]:
from langchain_core.documents import Document

# 선택된 chunk 를 Document 객체로 감쌉니다. (map_refine_chain 입력 형식)
selected_docs = [Document(page_content=split_docs[idx]) for idx in selected_indices]

for doc in selected_docs[:2]:
    print(doc.page_content[:200], "\n---")

## 대표 chunk 로 Map-Refine 요약

03 노트북과 같은 `map_refine_chain` 을 정의해 사용합니다.

### 프롬프트 가져오기 (LangSmith Hub)

> ⚠️ **레거시 변경점**
> 예전 코드의 `from langchain import hub` / `hub.pull(...)` 은 LangChain 1.x 에서 `langchain_classic` 으로 밀려났고,
> 최신 `langsmith` 는 **다른 사람(owner)의 공개 프롬프트**를 기본적으로 막습니다.
> 그래서 `langsmith.Client().pull_prompt(..., dangerously_pull_public_prompt=True)` 로 직접 가져옵니다.
> (`teddynote` 튜토리얼 프롬프트처럼 **신뢰할 수 있는 프롬프트에만** 사용하세요.)

In [ ]:
from langsmith import Client

client = Client()


def pull_prompt(name: str):
    """LangSmith Hub 에서 공개 프롬프트를 가져옵니다. (hub.pull 대체)"""
    return client.pull_prompt(name, dangerously_pull_public_prompt=True)

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
from langchain_core.runnables import chain


@chain
def map_refine_chain(docs):
    """Document 리스트를 받아 Map-Refine 방식으로 요약합니다."""
    # 1) Map: 각 문서를 병렬로 요약
    map_chain = pull_prompt("teddynote/map-summary-prompt") | llm | StrOutputParser()
    doc_summaries = map_chain.batch(
        [{"documents": doc.page_content, "language": "Korean"} for doc in docs]
    )

    # 2) Refine: 첫 요약에서 시작해 나머지 요약을 하나씩 반영하며 다듬기 (순차 실행)
    refine_chain = pull_prompt("teddynote/refine-prompt") | llm | StrOutputParser()
    previous_summary = doc_summaries[0]
    for i, current_summary in enumerate(doc_summaries[1:], start=2):
        previous_summary = refine_chain.invoke(
            {
                "previous_summary": previous_summary,
                "current_summary": current_summary,
                "language": "Korean",
            }
        )
        print(f"[refine {i}/{len(doc_summaries)}] 완료")

    return previous_summary

In [ ]:
refined_summary = map_refine_chain.invoke(selected_docs)
print("\n============== [최종 요약] ==============\n")
print(refined_summary)